# COMP9444 FOMC: Complete LSTM/BiLSTM/TCN Experiment Record

This notebook is the single execution and evidence record for the recurrent-model work. It displays the fixed baseline, all retained tuning attempts, unsuccessful candidates, the additional LSTM+TCN hybrid, three-seed test metrics, confusion matrices, learning curves, and error cases.

The notebook first uses cached canonical outputs. If an expected output is missing, the setup cell automatically runs the corresponding script with the current Python kernel and then continues. Therefore, executing all cells from top to bottom is sufficient; a clean run can be long on CPU.

## Protocol and paper alignment

The task is sentence-level three-class FOMC policy-stance classification: 0 Dovish, 1 Hawkish, and 2 Neutral. The shared protocol uses the fixed stratified validation splits, validation weighted F1 for selection, and the three held-out seeds 5768, 78516, and 944601 for the final report.

Shah et al. (2023), Section 4.2, describes a single-layer LSTM/BiLSTM baseline with word tokenisation, padding masking, dropout, and cross-entropy training. The LSTM+TCN model below is deliberately shown as an additional hybrid architecture ablation. It does not replace the required baselines and is not claimed as pretrained-model fine-tuning.

In [ ]:

from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display

def locate_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "lstm_bilstm.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the COMP9444_26T2_FOMC_Analysis repository.")

REPO_ROOT = locate_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.lstm_bilstm import (
    LABEL_NAMES,
    choose_device,
    create_or_load_validation_split,
    load_seed,
    resolve_data_dir,
    tokenize,
    write_overlap_audit,
)

DATA_DIR = resolve_data_dir(REPO_ROOT.parent / "FOMC_dataset_checkpoint")
SPLIT_DIR = REPO_ROOT / "data" / "splits"
SEEDS = (5768, 78516, 944601)
DEVICE = choose_device("auto")
AUTO_RUN_MISSING = True

RESULT_ROOTS = {
    "baseline": REPO_ROOT / "results" / "baseline_unsw",
    "bilstm_tuning": REPO_ROOT / "results" / "lstm_bilstm_tuning_v1",
    "lstm_v2": REPO_ROOT / "results" / "lstm_v2",
    "lstm_v3": REPO_ROOT / "results" / "lstm_v3",
    "lstm_tcn": REPO_ROOT / "results" / "lstm_tcn_tuning_v1",
    "lstm_three_paths_v2": REPO_ROOT / "results" / "lstm_three_paths_v2",
}
SUMMARY_RELATIVE = Path("final") / "lstm_bilstm_summary.csv"

JOBS = [
    (
        "same-environment baseline",
        RESULT_ROOTS["baseline"] / "lstm_bilstm_summary.csv",
        ["scripts/rerun_baseline_unsw.py"],
    ),
    (
        "LSTM/BiLSTM tuning",
        RESULT_ROOTS["bilstm_tuning"] / SUMMARY_RELATIVE,
        ["scripts/tune_lstm_bilstm.py", "--device", "cpu"],
    ),
    (
        "LSTM second-round attempt",
        RESULT_ROOTS["lstm_v2"] / SUMMARY_RELATIVE,
        ["scripts/tune_lstm_v2.py", "--device", "cpu"],
    ),
    (
        "LSTM third-round attempt",
        RESULT_ROOTS["lstm_v3"] / SUMMARY_RELATIVE,
        ["scripts/tune_lstm_v3.py", "--device", "cpu"],
    ),
    (
        "LSTM+TCN hybrid search",
        RESULT_ROOTS["lstm_tcn"] / SUMMARY_RELATIVE,
        ["scripts/tune_lstm_tcn.py", "--device", "cpu"],
    ),
    (
        "LSTM/BiLSTM/TCN second-stage v2 search",
        REPO_ROOT / "results" / "lstm_three_paths_v2" / "three_path_comparison.csv",
        ["scripts/tune_lstm_three_paths_v2.py", "--device", "cpu"],
    ),
]

def ensure_experiment_outputs():
    missing = [(label, marker, command) for label, marker, command in JOBS if not marker.exists()]
    if not missing:
        print("All canonical experiment outputs are present; the notebook will display the cached runs.")
        return
    if not AUTO_RUN_MISSING:
        missing_text = "\n".join(f"- {label}: {marker}" for label, marker, _ in missing)
        raise FileNotFoundError("Missing experiment outputs:\n" + missing_text)
    for label, marker, command in missing:
        print(f"\nRunning missing experiment: {label}")
        print("Command:", sys.executable, *command)
        subprocess.run([sys.executable, *command], cwd=REPO_ROOT, check=True)
        if not marker.exists():
            raise FileNotFoundError(f"Experiment finished without expected output: {marker}")

ensure_experiment_outputs()
print("Repository:", REPO_ROOT)
print("Data:", DATA_DIR)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
display(Markdown("Labels: " + ", ".join(f"{key} = {value}" for key, value in LABEL_NAMES.items())))


In [ ]:

dataset_rows = []
audit_rows = []
for seed in SEEDS:
    source_train, source_test = load_seed(DATA_DIR, seed)
    train_part, validation_part = create_or_load_validation_split(
        source_train, seed, SPLIT_DIR
    )
    audit = write_overlap_audit(
        source_train, source_test, seed, SPLIT_DIR.parent / "audits"
    )
    dataset_rows.append(
        {
            "seed": seed,
            "source_train_rows": len(source_train),
            "effective_train_rows": len(train_part),
            "validation_rows": len(validation_part),
            "test_rows": len(source_test),
            "max_train_tokens": max(len(tokenize(record.sentence)) for record in train_part),
        }
    )
    audit_rows.append(
        {
            "seed": seed,
            "index_overlap": audit["index_overlap_count"],
            "normalised_text_overlap": audit["normalised_sentence_overlap_count"],
            "duplicate_train_sentences": audit["duplicate_sentences_within_train"],
            "duplicate_test_sentences": audit["duplicate_sentences_within_test"],
        }
    )

display(Markdown("## Dataset and split audit"))
display(pd.DataFrame(dataset_rows))
display(pd.DataFrame(audit_rows))


## Historical tuning attempts

The search tables below are kept as evidence. A lower-scoring candidate is not deleted: it is labelled failed to improve the control or not selected. This makes the Notebook show why the final configuration was chosen instead of presenting only a successful result.

In [ ]:

SEARCH_SOURCES = [
    (
        "paper reconstruction search",
        REPO_ROOT / "results" / "lstm_bilstm_paper_optimisation" / "paper_search",
    ),
    (
        "legacy optimisation search",
        REPO_ROOT / "results" / "lstm_bilstm_paper_optimisation" / "optimisation_search",
    ),
    (
        "LSTM/BiLSTM tuning v1",
        RESULT_ROOTS["bilstm_tuning"] / "validation_search",
    ),
    (
        "LSTM tuning v2",
        RESULT_ROOTS["lstm_v2"] / "validation_search",
    ),
    (
        "LSTM tuning v3",
        RESULT_ROOTS["lstm_v3"] / "validation_search",
    ),
    (
        "LSTM+TCN tuning",
        RESULT_ROOTS["lstm_tcn"] / "validation_search",
    ),
]

history_frames = []
for group_name, directory in SEARCH_SOURCES:
    for path in sorted(directory.glob("*validation_search.csv")):
        frame = pd.read_csv(path)
        if "validation_weighted_f1" not in frame.columns:
            frame["validation_weighted_f1"] = frame["validation_weighted_f1_mean"]
        if "validation_macro_f1" not in frame.columns:
            frame["validation_macro_f1"] = frame["validation_macro_f1_mean"]
        if "candidate" not in frame.columns:
            frame["candidate"] = frame.get("run", pd.Series(range(1, len(frame) + 1)))
        frame["attempt_group"] = group_name
        frame["search_file"] = str(path.relative_to(REPO_ROOT))
        history_frames.append(frame)

history = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()
if history.empty:
    display(Markdown("No validation search CSV files were found."))
else:
    status_frames = []
    for group_name, group in history.groupby("attempt_group", sort=False):
        group = group.copy()
        best_index = group["validation_weighted_f1"].idxmax()
        control_rows = group[
            group["candidate"].astype(str).str.contains("control|legacy", case=False, regex=True)
        ]
        control_score = (
            control_rows["validation_weighted_f1"].max()
            if not control_rows.empty
            else group["validation_weighted_f1"].min()
        )
        group["attempt_status"] = np.select(
            [
                group.index == best_index,
                group["validation_weighted_f1"] < control_score - 1e-9,
            ],
            ["selected", "failed_to_improve_control"],
            default="not_selected",
        )
        status_frames.append(group)
    history = pd.concat(status_frames, ignore_index=True)
    history_view = history[
        [
            "attempt_group",
            "model",
            "candidate",
            "attempt_status",
            "validation_weighted_f1",
            "validation_macro_f1",
            "parameter_count",
        ]
    ].sort_values(
        ["attempt_group", "model", "validation_weighted_f1"],
        ascending=[True, True, False],
    )
    display(Markdown(
        "## Visible tuning history\n\n"
        "The failed or not-selected candidates are intentionally kept. "
        "A candidate is labelled failed when its validation weighted F1 is below "
        "that search's control configuration; test data was not used for this label."
    ))
    display(history_view.reset_index(drop=True))


In [ ]:

def read_summary(root):
    paths = [
        root / "lstm_bilstm_summary.csv",
        root / SUMMARY_RELATIVE,
    ]
    for path in paths:
        if path.exists():
            return pd.read_csv(path)
    return pd.DataFrame()

baseline_summary = read_summary(RESULT_ROOTS["baseline"])
tuned_summary = read_summary(RESULT_ROOTS["bilstm_tuning"])
v2_summary = read_summary(RESULT_ROOTS["lstm_v2"])
v3_summary = read_summary(RESULT_ROOTS["lstm_v3"])
tcn_summary = read_summary(RESULT_ROOTS["lstm_tcn"])
legacy_paper_root = REPO_ROOT / "results" / "lstm_bilstm_paper_optimisation"
paper_summary = read_summary(legacy_paper_root / "paper_replication")
legacy_optimised_summary = read_summary(legacy_paper_root / "optimised")

def aggregate_rows(summary, experiment_name, model_name, protocol_note):
    group = summary[summary["model"] == model_name].copy()
    if group.empty:
        return []
    return [
        {
            "experiment": experiment_name,
            "model": model_name,
            "protocol": protocol_note,
            "weighted_f1_mean": group["test_weighted_f1"].mean(),
            "weighted_f1_std": group["test_weighted_f1"].std(),
            "macro_f1_mean": group["test_macro_f1"].mean(),
            "macro_f1_std": group["test_macro_f1"].std(),
            "accuracy_mean": group["test_accuracy"].mean(),
            "parameter_count_mean": group["parameter_count"].mean(),
        }
    ]

comparison_rows = []
comparison_rows += aggregate_rows(
    baseline_summary, "same-environment baseline", "lstm", "current baseline"
)
comparison_rows += aggregate_rows(
    baseline_summary, "same-environment baseline", "bilstm", "current baseline"
)
comparison_rows += aggregate_rows(
    tuned_summary, "tuned LSTM", "lstm", "validation-selected LSTM"
)
comparison_rows += aggregate_rows(
    tuned_summary, "tuned BiLSTM", "bilstm", "validation-selected BiLSTM"
)
comparison_rows += aggregate_rows(
    v2_summary, "LSTM v2", "lstm", "failed improvement attempt"
)
comparison_rows += aggregate_rows(
    v3_summary, "LSTM v3", "lstm", "no-gain control attempt"
)
comparison_rows += aggregate_rows(
    tcn_summary, "LSTM+TCN", "lstm_tcn", "additional hybrid architecture ablation"
)
comparison_rows += aggregate_rows(
    paper_summary, "paper reconstruction (legacy)", "lstm", "different paper-matched protocol"
)
comparison_rows += aggregate_rows(
    paper_summary, "paper reconstruction (legacy)", "bilstm", "different paper-matched protocol"
)
comparison_rows += aggregate_rows(
    legacy_optimised_summary, "legacy paper optimisation", "lstm", "different legacy protocol"
)
comparison_rows += aggregate_rows(
    legacy_optimised_summary, "legacy paper optimisation", "bilstm", "different legacy protocol"
)

comparison = pd.DataFrame(comparison_rows)
display(Markdown("## Final three-seed comparison"))
display(comparison.round(4))


In [ ]:

def seed_metrics(summary, label, model_name):
    frame = summary[summary["model"] == model_name][
        ["seed", "test_weighted_f1", "test_macro_f1", "test_accuracy"]
    ].copy()
    return frame.rename(
        columns={
            "test_weighted_f1": f"{label} weighted F1",
            "test_macro_f1": f"{label} macro F1",
            "test_accuracy": f"{label} accuracy",
        }
    )

seed_comparison = pd.DataFrame({"seed": list(SEEDS)})
for frame in (
    seed_metrics(baseline_summary, "LSTM baseline", "lstm"),
    seed_metrics(tuned_summary, "BiLSTM tuned", "bilstm"),
    seed_metrics(tcn_summary, "LSTM+TCN", "lstm_tcn"),
):
    seed_comparison = seed_comparison.merge(frame, on="seed", how="left")

seed_comparison["TCN weighted F1 minus LSTM"] = (
    seed_comparison["LSTM+TCN weighted F1"]
    - seed_comparison["LSTM baseline weighted F1"]
)
seed_comparison["TCN weighted F1 minus BiLSTM"] = (
    seed_comparison["LSTM+TCN weighted F1"]
    - seed_comparison["BiLSTM tuned weighted F1"]
)
display(Markdown("## Per-seed comparison"))
display(seed_comparison.round(4))


In [ ]:

TCN_OUTPUT = RESULT_ROOTS["lstm_tcn"] / "validation_search"
tcn_config = json.loads((TCN_OUTPUT / "best_config.json").read_text())
display(Markdown("## Selected LSTM+TCN configuration"))
display(pd.DataFrame([tcn_config["config"]]).T.rename(columns={0: "value"}))

from src.lstm_tcn import LSTMTCNClassifier

torch.manual_seed(9444)
smoke_model = LSTMTCNClassifier(
    vocab_size=128,
    embedding_dim=16,
    hidden_size=12,
    dropout=0.3,
    lstm_layers=2,
    tcn_channels=10,
    tcn_kernel_size=3,
    tcn_layers=3,
    pooling="mean_max",
)
smoke_ids = torch.randint(0, 128, (4, 20))
smoke_lengths = torch.tensor([20, 15, 8, 1])
smoke_logits = smoke_model(smoke_ids, smoke_lengths)
assert tuple(smoke_logits.shape) == (4, 3)
display(pd.DataFrame([{
    "test": "LSTM+TCN forward-pass smoke test",
    "input_shape": tuple(smoke_ids.shape),
    "output_shape": tuple(smoke_logits.shape),
    "multi_layer_configuration": True,
    "parameter_count": sum(parameter.numel() for parameter in smoke_model.parameters()),
    "status": "PASS",
}]))


In [ ]:

display(Markdown("## Validation search curve"))
if not history.empty:
    tcn_history = history[history["attempt_group"] == "LSTM+TCN tuning"].copy()
    if not tcn_history.empty:
        tcn_history["label"] = tcn_history["candidate"].astype(str)
        tcn_history = tcn_history.sort_values("validation_weighted_f1", ascending=True)
        figure, axis = plt.subplots(figsize=(10, 5))
        axis.barh(tcn_history["label"], tcn_history["validation_weighted_f1"], color="#2f6f8f")
        axis.set_xlabel("Seed 5768 validation weighted F1")
        axis.set_title("LSTM+TCN candidate history, including non-selected attempts")
        axis.grid(axis="x", alpha=0.25)
        figure.tight_layout()
        plt.show()

def show_image(path):
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Missing image:", path)

for label, root in (
    ("Same-environment baseline", RESULT_ROOTS["baseline"]),
    ("Tuned LSTM/BiLSTM", RESULT_ROOTS["bilstm_tuning"] / "final"),
    ("LSTM+TCN", RESULT_ROOTS["lstm_tcn"] / "final"),
):
    display(Markdown(f"### {label} aggregate plots"))
    show_image(root / "lstm_bilstm_weighted_f1.png")
    show_image(root / "lstm_bilstm_macro_f1.png")


In [ ]:

def show_run_artifacts(title, root, model_name, seed):
    run_dir = root / model_name / f"seed_{seed}"
    display(Markdown(f"## {title}: {model_name}, seed {seed}"))
    metrics_path = run_dir / "metrics.json"
    report_path = run_dir / "classification_report.csv"
    matrix_path = run_dir / "confusion_matrix_normalized.png"
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
        display(pd.DataFrame([metrics["test"]]))
    if report_path.exists():
        display(pd.read_csv(report_path))
    show_image(matrix_path)

best_tcn_seed = int(
    tcn_summary.sort_values("test_weighted_f1", ascending=False).iloc[0]["seed"]
)
best_bilstm_seed = int(
    tuned_summary[tuned_summary["model"] == "bilstm"]
    .sort_values("test_weighted_f1", ascending=False)
    .iloc[0]["seed"]
)
for seed in SEEDS:
    show_run_artifacts(
        "LSTM+TCN confusion matrix and report",
        RESULT_ROOTS["lstm_tcn"] / "final",
        "lstm_tcn",
        seed,
    )
show_run_artifacts(
    "Best tuned BiLSTM confusion matrix and report",
    RESULT_ROOTS["bilstm_tuning"] / "final",
    "bilstm",
    best_bilstm_seed,
)


In [ ]:

for title, root, model_name, seed in (
    (
        "LSTM+TCN learning curve and errors",
        RESULT_ROOTS["lstm_tcn"] / "final",
        "lstm_tcn",
        best_tcn_seed,
    ),
    (
        "Best tuned BiLSTM learning curve and errors",
        RESULT_ROOTS["bilstm_tuning"] / "final",
        "bilstm",
        best_bilstm_seed,
    ),
):
    run_dir = root / model_name / f"seed_{seed}"
    display(Markdown(f"## {title}"))
    show_image(run_dir / "learning_curve.png")
    error_path = run_dir / "selected_error_cases.csv"
    if error_path.exists():
        display(pd.read_csv(error_path)[
            ["true_label_name", "predicted_label_name", "category", "sentence"]
        ])


In [ ]:

tcn_row = comparison[comparison["model"] == "lstm_tcn"].iloc[0]
lstm_row = comparison[
    (comparison["experiment"] == "same-environment baseline")
    & (comparison["model"] == "lstm")
].iloc[0]
bilstm_row = comparison[
    (comparison["experiment"] == "tuned BiLSTM")
    & (comparison["model"] == "bilstm")
].iloc[0]
tcn_wins = int(
    (
        seed_comparison["LSTM+TCN weighted F1"]
        > seed_comparison[["LSTM baseline weighted F1", "BiLSTM tuned weighted F1"]].max(axis=1)
    ).sum()
)
display(Markdown(
    "## Conclusions\n\n"
    f"The selected LSTM+TCN reaches weighted F1 "
    f"{tcn_row['weighted_f1_mean']:.4f} +/- {tcn_row['weighted_f1_std']:.4f} "
    f"and macro F1 {tcn_row['macro_f1_mean']:.4f} +/- {tcn_row['macro_f1_std']:.4f}. "
    f"It improves the three-seed weighted-F1 mean over the same-environment LSTM "
    f"by {tcn_row['weighted_f1_mean'] - lstm_row['weighted_f1_mean']:.4f} "
    f"and over the tuned BiLSTM by {tcn_row['weighted_f1_mean'] - bilstm_row['weighted_f1_mean']:.4f}. "
    f"It beats both reference models on {tcn_wins} of the {len(SEEDS)} seeds, "
    "so the improvement is useful on average but not universal.\n\n"
    "The paper and project plan use a single-layer LSTM/BiLSTM as the exact baseline. "
    "The two-layer LSTM candidate is therefore reported as an ablation; it was not selected. "
    "LSTM+TCN is an additional hybrid architecture experiment, not a pretrained-model fine-tuning result.\n\n"
    "All candidate selection used the seed 5768 validation split only. "
    "The three held-out test sets were evaluated after freezing the selected configuration."
))


## Reproducibility checklist

- The model-selection seed is 5768 and the final test seeds are 5768, 78516, and 944601.
- Baseline and LSTM/BiLSTM tuning use the shared source module in src/lstm_bilstm.py.
- LSTM+TCN is implemented in src/lstm_tcn.py and tuned by scripts/tune_lstm_tcn.py.
- Every final seed has a configuration file, metrics, classification report, raw and normalised confusion matrices, predictions, error analysis, learning curve, and saved checkpoint.
- The v2 results directory is results/lstm_three_paths_v2; the prior historical directories remain visible in the comparison cells.

## Second-stage three-path experiment (v2)

This section is the current controlled experiment for only LSTM, BiLSTM, and LSTM+TCN. Candidate selection used the validation split for seed 5768 only. After selection, the configuration was frozen and evaluated on seeds 5768, 78516, and 944601. The main comparison is the mean and standard deviation of weighted F1; macro F1, accuracy, per-class F1, parameter count, and training cost are also retained.

In [ ]:
# add
V2_ROOT = REPO_ROOT / "results" / "lstm_three_paths_v2"
V2_MODELS = {"lstm": "LSTM v2", "bilstm": "BiLSTM v2", "lstm_tcn": "LSTM+TCN v2"}
v2_comparison = pd.read_csv(V2_ROOT / "three_path_comparison.csv")
v2_columns = ["model", "baseline_reference", "weighted_f1_mean", "weighted_f1_std", "macro_f1_mean", "macro_f1_std", "accuracy_mean", "dovish_f1_mean", "hawkish_f1_mean", "neutral_f1_mean", "parameter_count_mean", "mean_training_seconds", "weighted_f1_delta_vs_baseline", "wins_vs_baseline", "status"]
display(Markdown("### Frozen v2 comparison"))
display(v2_comparison[v2_columns].round(4))
selected = json.loads((V2_ROOT / "validation_search" / "selected_configs.json").read_text())
selected_rows = [{"model": model, **config} for model, config in selected["configs"].items()]
display(Markdown("### Configurations selected on validation seed 5768"))
display(pd.DataFrame(selected_rows).T)
search = pd.read_csv(V2_ROOT / "validation_search" / "three_paths_validation_search.csv")
search_columns = ["model", "candidate", "validation_weighted_f1", "validation_macro_f1", "parameter_count", "attempt_status"]
display(Markdown("### Complete candidate history, including failures and non-selected candidates"))
display(search[search_columns].sort_values(["model", "validation_weighted_f1"], ascending=[True, False]).round(4))
glove_status = json.loads((V2_ROOT / "glove_status.json").read_text())
display(Markdown("GloVe status: " + glove_status["status"] + ". " + glove_status["reason"]))
#done


In [ ]:
# add
v2_final = pd.read_csv(V2_ROOT / "final" / "lstm_bilstm_summary.csv")
display(Markdown("### Frozen-model per-seed evidence"))
display(v2_final.round(4))
# add
for figure_name in ["three_paths_weighted_f1.png", "three_paths_macro_f1.png", "three_paths_per_class_f1.png", "three_paths_cost.png"]:
    show_image(V2_ROOT / figure_name)
#done
for model, title in V2_MODELS.items():
    display(Markdown(f"#### {title}"))
    for seed in SEEDS:
        run_dir = V2_ROOT / "final" / model / f"seed_{seed}"
        metrics = json.loads((run_dir / "metrics.json").read_text())
        test_metrics = metrics.get("test", metrics)
        display(Markdown(f"Seed {seed}: weighted F1 = {test_metrics['weighted_f1']:.4f}, macro F1 = {test_metrics['macro_f1']:.4f}, accuracy = {test_metrics['accuracy']:.4f}"))
        display(pd.read_csv(run_dir / "classification_report.csv").round(4))
        display(pd.read_csv(run_dir / "confusion_matrix_raw.csv"))
        show_image(run_dir / "confusion_matrix_normalized.png")
        show_image(run_dir / "learning_curve.png")
        error_path = run_dir / "selected_error_cases.csv"
        if error_path.exists():
            display(pd.read_csv(error_path).head(20))
#done
